# Heart Disease Prediction - Model Training Pipeline
This notebook covers data preprocessing, model training, evaluation, and saving the best model for deployment using Flask.

## 1. Import Libraries
Import all required libraries for data processing, modeling, and evaluation.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
import joblib

## 2. Load Dataset
Load the heart disease dataset from the data directory.

In [ ]:
df = pd.read_csv('../data/heart_disease_uci.csv')
df = df.copy()

## 4. Feature Encoding
Convert categorical variables into numerical format using One-Hot Encoding and Label Encoding.

In [ ]:
num_cols = ['trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
cat_cols = ['sex', 'dataset', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

# Missing values
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Remove outliers
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    df = df[(df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)]

df.reset_index(drop=True, inplace=True)

In [ ]:
nominal_cols = ['cp', 'thal', 'restecg', 'slope', 'dataset']
binary_cols = ['sex', 'fbs', 'exang']

df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

## 5. Feature Scaling and Train-Test Split
Scale numerical features and split the dataset into training and testing sets.

In [ ]:
target = 'num'
features = [col for col in df.columns if col not in ['id', 'num']]

X = df[features]
y = df[target]

# Scaling
num_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 6. Model Training and Hyperparameter Tuning
Train multiple models and tune their hyperparameters using GridSearchCV.

In [ ]:
results = {
    "RF": rf_grid.best_score_,
    "LR": lr_grid.best_score_,
    "XGB": xgb_grid.best_score_
}

best_model = max(
    [rf_grid, lr_grid, xgb_grid],
    key=lambda x: x.best_score_
)

print("Best Model:", best_model.best_estimator_)

## 7. Save the Best Model
Save the trained model to the models directory for later use in the Flask application.

In [ ]:
joblib.dump(best_model.best_estimator_, "../models/heart_model.pkl")